In [1]:
import duckdb
from pathlib import Path

DATA = Path("data")
con = duckdb.connect()

tables = ["application_train", "application_test", "bureau", "bureau_balance",
          "previous_application", "POS_CASH_balance", "credit_card_balance",
          "installments_payments"]

for t in tables:
    path = (DATA / f"{t}.csv").as_posix()
    rows = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    cols = len(con.execute(f"SELECT * FROM read_csv_auto('{path}') LIMIT 0").description)
    size_mb = Path(path).stat().st_size / 1_000_000
    print(f"{t:22s} rows={rows:>11,}  cols={cols:>4}  size={size_mb:7.1f} MB")

application_train      rows=    307,511  cols= 122  size=  166.1 MB
application_test       rows=     48,744  cols= 121  size=   26.6 MB
bureau                 rows=  1,716,428  cols=  17  size=  170.0 MB
bureau_balance         rows= 27,299,925  cols=   3  size=  375.6 MB
previous_application   rows=  1,670,214  cols=  37  size=  405.0 MB
POS_CASH_balance       rows= 10,001,358  cols=   8  size=  392.7 MB
credit_card_balance    rows=  3,840,312  cols=  23  size=  424.6 MB
installments_payments  rows= 13,605,401  cols=   8  size=  723.1 MB


In [2]:
train = (DATA / "application_train.csv").as_posix()

result = con.execute(f"""
    SELECT TARGET,
           COUNT(*) AS customers,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percent
    FROM read_csv_auto('{train}')
    GROUP BY TARGET
    ORDER BY TARGET
""").df()

result

,TARGET,customers,percent
0,0,282686,91.93
1,1,24825,8.07


In [3]:
for t in tables:
    path = (DATA / f"{t}.csv").as_posix()
    cols = [c[0] for c in con.execute(f"SELECT * FROM read_csv_auto('{path}') LIMIT 0").description]
    keys = [c for c in cols if c.startswith("SK_ID")]
    if "SK_ID_CURR" in cols:
        n = con.execute(f"SELECT COUNT(DISTINCT SK_ID_CURR) FROM read_csv_auto('{path}')").fetchone()[0]
        print(f"{t:22s} keys={keys}  unique applicants={n:,}")
    else:
        print(f"{t:22s} keys={keys}  (no SK_ID_CURR)")

application_train      keys=['SK_ID_CURR']  unique applicants=307,511
application_test       keys=['SK_ID_CURR']  unique applicants=48,744
bureau                 keys=['SK_ID_CURR', 'SK_ID_BUREAU']  unique applicants=305,811
bureau_balance         keys=['SK_ID_BUREAU']  (no SK_ID_CURR)
previous_application   keys=['SK_ID_PREV', 'SK_ID_CURR']  unique applicants=338,857
POS_CASH_balance       keys=['SK_ID_PREV', 'SK_ID_CURR']  unique applicants=337,252
credit_card_balance    keys=['SK_ID_PREV', 'SK_ID_CURR']  unique applicants=103,558
installments_payments  keys=['SK_ID_PREV', 'SK_ID_CURR']  unique applicants=339,587


In [4]:
history_tables = ["bureau", "previous_application", "POS_CASH_balance",
                  "credit_card_balance", "installments_payments"]

for t in history_tables:
    path = (DATA / f"{t}.csv").as_posix()
    n = con.execute(f"""
        SELECT COUNT(*) FROM read_csv_auto('{train}') a
        WHERE a.SK_ID_CURR IN (SELECT SK_ID_CURR FROM read_csv_auto('{path}'))
    """).fetchone()[0]
    print(f"{t:22s} {n:>9,} of 307,511 have history  ({100*n/307511:.1f}%)")

bureau                   263,491 of 307,511 have history  (85.7%)
previous_application     291,057 of 307,511 have history  (94.6%)
POS_CASH_balance         289,444 of 307,511 have history  (94.1%)
credit_card_balance       86,905 of 307,511 have history  (28.3%)
installments_payments    291,643 of 307,511 have history  (94.8%)


In [5]:
for t in history_tables:
    path = (DATA / f"{t}.csv").as_posix()
    res = con.execute(f"""
        SELECT
          CASE WHEN a.SK_ID_CURR IN (SELECT SK_ID_CURR FROM read_csv_auto('{path}'))
               THEN 'has history' ELSE 'no history' END AS grp,
          COUNT(*) AS people,
          ROUND(100.0 * AVG(a.TARGET), 2) AS default_pct
        FROM read_csv_auto('{train}') a
        GROUP BY grp ORDER BY grp
    """).fetchall()
    print(t)
    for grp, people, pct in res:
        print(f"   {grp:12s} {people:>8,} people   default = {pct}%")

bureau
   has history   263,491 people   default = 7.73%
   no history     44,020 people   default = 10.12%
previous_application
   has history   291,057 people   default = 8.19%
   no history     16,454 people   default = 5.96%
POS_CASH_balance
   has history   289,444 people   default = 8.16%
   no history     18,067 people   default = 6.66%
credit_card_balance
   has history    86,905 people   default = 8.67%
   no history    220,606 people   default = 7.84%
installments_payments
   has history   291,643 people   default = 8.19%
   no history     15,868 people   default = 5.98%


In [6]:
import pandas as pd

desc = pd.read_csv(DATA / "HomeCredit_columns_description.csv", encoding="cp1252")

key_cols = ["TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
            "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

app_desc = desc[desc["Table"].str.contains("application")]
pd.set_option("display.max_colwidth", 120)
app_desc[app_desc["Row"].isin(key_cols)][["Row", "Description", "Special"]]

,Row,Description,Special
1,TARGET,Target variable (1 - client with payment difficulties: he/she had late payment more than X days on at least one of t...,NaN
7,AMT_INCOME_TOTAL,Income of the client,NaN
8,AMT_CREDIT,Credit amount of the loan,NaN
9,AMT_ANNUITY,Loan annuity,NaN
17,DAYS_BIRTH,Client's age in days at the time of application,time only relative to the application
18,DAYS_EMPLOYED,How many days before the application the person started current employment,time only relative to the application
41,EXT_SOURCE_1,Normalized score from external data source,normalized
42,EXT_SOURCE_2,Normalized score from external data source,normalized
43,EXT_SOURCE_3,Normalized score from external data source,normalized
176,AMT_ANNUITY,Annuity of previous application,NaN


In [7]:
main_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
             "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

rows = []
for c in main_cols:
    r = con.execute(f"""
        SELECT MIN({c}), MAX({c}), ROUND(AVG({c}), 2),
               ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1)
        FROM read_csv_auto('{train}')
    """).fetchone()
    rows.append([c, *r])

pd.DataFrame(rows, columns=["column", "min", "max", "average", "missing_%"])

,column,min,max,average,missing_%
0,AMT_INCOME_TOTAL,2.565000e+04,1.170000e+08,168797.92,0.0
1,AMT_CREDIT,4.500000e+04,4.050000e+06,599026.00,0.0
2,AMT_ANNUITY,1.615500e+03,2.580255e+05,27108.57,0.0
3,DAYS_BIRTH,-2.522900e+04,-7.489000e+03,-16037.00,0.0
4,DAYS_EMPLOYED,-1.791200e+04,3.652430e+05,63815.05,0.0
5,EXT_SOURCE_1,1.456813e-02,9.626928e-01,0.50,56.4
6,EXT_SOURCE_2,8.173617e-08,8.549997e-01,0.51,0.2
7,EXT_SOURCE_3,5.272652e-04,8.960095e-01,0.51,19.8


In [8]:
con.execute(f"""
    SELECT
      CASE WHEN DAYS_EMPLOYED = 365243 THEN 'fake 365243' ELSE 'normal' END AS grp,
      COUNT(*) AS people,
      ROUND(100.0 * AVG(TARGET), 2) AS default_pct,
      ROUND(AVG(-DAYS_BIRTH) / 365.25, 1) AS avg_age
    FROM read_csv_auto('{train}')
    GROUP BY grp
""").df()

,grp,people,default_pct,avg_age
0,normal,252137,8.66,40.4
1,fake 365243,55374,5.40,59.7


In [9]:
con.execute(f"""
    SELECT NAME_INCOME_TYPE, COUNT(*) AS people
    FROM read_csv_auto('{train}')
    WHERE DAYS_EMPLOYED = 365243
    GROUP BY NAME_INCOME_TYPE
    ORDER BY people DESC
""").df()

,NAME_INCOME_TYPE,people
0,Pensioner,55352
1,Unemployed,22


In [10]:
con.execute(f"""
    SELECT SK_ID_CURR, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY,
           NAME_INCOME_TYPE, TARGET
    FROM read_csv_auto('{train}')
    ORDER BY AMT_INCOME_TOTAL DESC
    LIMIT 5
""").df()

,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,TARGET
0,114967,117000000.0,562491.0,26194.5,Working,1
1,336147,18000090.0,675000.0,69295.5,Commercial associate,0
2,385674,13500000.0,1400503.5,130945.5,Commercial associate,0
3,190160,9000000.0,1431531.0,132601.5,Working,0
4,252084,6750000.0,790830.0,52978.5,Working,0


## Stage 2 findings: understanding the data

- **8 tables, ~58 million rows.** Profiled with DuckDB to stay within 16 GB RAM.
- **One row per applicant** in `application_train` (307,511 unique `SK_ID_CURR`).
- **Imbalanced target:** 8.07% defaulted (24,825). Accuracy is misleading → use ROC-AUC and PR-AUC.
- **Missing history carries meaning:** no bureau record → 10.12% default vs 7.73% with a record.
- **`DAYS_EMPLOYED = 365243`** is a placeholder: 55,374 people (18.0%), 99.96% pensioners. Default 5.40% vs 8.66%. Plan: set to missing + add a flag.
- **Income outlier:** one applicant reports 117,000,000 income with a 562,491 loan and defaulted. Likely an entry error. Plan: cap extreme incomes.
- **`EXT_SOURCE_1`** is 56.4% missing.
- `TARGET = 1` means **payment difficulties** (late payments), not necessarily total non-repayment.